# Kolmogorov Navier–Stokes dataset — **validated GPU generation only**

**Purpose:** generate 300 independent, spin-up Kolmogorov-flow **vorticity** trajectories on a GPU and export a single `kolmogorov_dataset.pt` to Google Drive. **No neural-network training, AWFNO, or K sweep is included.**

We preserve the original dataset definition:

\[
\partial_t\omega+\mathbf u\cdot\nabla\omega=\nu\Delta\omega-4\cos(4y),\qquad
-\Delta\psi=\omega,\quad\mathbf u=(\partial_y\psi,-\partial_x\psi),
\]

on \([0,2\pi)^2\), \(\nu=1/500\), with two-thirds dealiasing, 128² collocation nodes, 20 time units of spin-up, and 51 snapshots at times 0 to 1 with spacing 0.02. Initial vorticity fields use **exactly the original NumPy seeded Gaussian filter**. Samples 0–199 are training, 200–229 validation, and 230–299 test.

**Reliability gates before generation:** (1) analytic forced-laminar solution test; (2) time-step halving, comparing the *same post-spin-up initial fields*; (3) 128² vs 256² spatial refinement, also from the same initial fields; (4) maximum Courant diagnostic; (5) data finiteness, evolving states, zero-mean vorticity, checksums and train/validation/test separation. If any gate fails, the notebook stops instead of silently publishing an unverified dataset.

The numerical method is a Fourier pseudospectral ETDRK4 solver with **real ETDRK4 coefficients** evaluated using stable entire-function Taylor series (\(\varphi_1,\varphi_2,\varphi_3\)); this avoids the spurious complex coefficients in the original script. CUDA float32/complex64 is used for generation; an independent float64/complex128 short-horizon consistency check is performed during validation. Passing these checks supports—but cannot mathematically prove—resolution or long-term statistical accuracy.

**Cost:** GPUs accelerate FFTs, but running 300 trajectories through 4,000 spin-up steps each is still substantial. Completed batches are saved to Drive and can be resumed. The notebook reports actual timings; no unverified runtime estimate is promised.

In [ ]:
#@title 1. Select GPU, mount Google Drive, and define the immutable dataset specification
from google.colab import drive
from pathlib import Path
import os, sys, json, time, math, shutil, hashlib, gc, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Change Runtime > Change runtime type > GPU, then rerun.'
DEVICE=torch.device('cuda')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',
      round(torch.cuda.get_device_properties(0).total_memory / 2**30,2))

# Use a new output folder so the original CPU-generated dataset is never overwritten.
DRIVE_FOLDER='/content/drive/MyDrive/Kolmogorov_NS_Re500_verified_gpu_v1' #@param {type:'string'}
ROOT=Path(DRIVE_FOLDER)
CHUNKS=ROOT/'chunks'
LOCAL=Path('/content/kolmogorov_ns_build')
for p in (ROOT,CHUNKS,LOCAL):p.mkdir(parents=True,exist_ok=True)

# Original generator's physical and reproducibility parameters (do not casually edit).
NX=128
RE=500.0
NU=1.0/RE
FORCING_MODE=4
DOMAIN=2.0*np.pi
DT=0.005
T_SPINUP=20.0
T_FORECAST=1.0
SAVE_EVERY=4
N_TRAIN,N_VAL,N_TEST=200,30,70
N_SAMPLES=N_TRAIN+N_VAL+N_TEST
BASE_SEED=1234
IC_AMPLITUDE=1.0
IC_FILTER_K0=6.0

# GPU execution controls: increasing batch size changes throughput, NOT PDE/dataset.
BATCH_SIZE=32 #@param {type:'integer'}
assert 1<=BATCH_SIZE<=64
SAVE_CHUNKS=True
REMOVE_CHUNKS_AFTER_FINAL_VERIFICATION=False #@param {type:'boolean'}

# Fail-closed numerical quality gates (can tighten for publication).
DT_REFINEMENT_MAX_REL=0.02
SPATIAL_REFINEMENT_MAX_REL=0.05
FP32_FP64_MAX_REL=0.003
MAX_RECORDED_CFL=0.9
MAX_SPECTRAL_CUTOFF_SHELL_FRACTION=0.02
CHECK_SAMPLES=2
MAX_GENERATION_HOURS=4.0 #@param {type:'number'}
# Abort before full generation if pilot predicts more than this GPU evolution time.

SPINUP_STEPS=round(T_SPINUP/DT)
FORECAST_STEPS=round(T_FORECAST/DT)
assert SPINUP_STEPS*DT==T_SPINUP
assert FORECAST_STEPS*DT==T_FORECAST
assert FORECAST_STEPS%SAVE_EVERY==0
NT=FORECAST_STEPS//SAVE_EVERY+1
TIMES=np.arange(NT,dtype=np.float64)*(DT*SAVE_EVERY)
assert len(TIMES)==51 and abs(TIMES[-1]-1.)<1e-12
X=np.arange(NX,dtype=np.float64)*(DOMAIN/NX)
Y=np.arange(NX,dtype=np.float64)*(DOMAIN/NX)
DATA_SHAPE=(N_SAMPLES,NX,NX,NT)
SPEC={'schema':'kolmogorov-vorticity-gpu-v1', 'shape':list(DATA_SHAPE),
      'Re':RE,'nu':NU,'forcing_mode':FORCING_MODE,'L':float(DOMAIN),
      'dt':DT,'T_spinup':T_SPINUP,'T_forecast':T_FORECAST,
      'save_every':SAVE_EVERY,'base_seed':BASE_SEED,
      'ic_amplitude':IC_AMPLITUDE,'ic_filter_k0':IC_FILTER_K0,
      'dtype':'float32','method':'ETDRK4-Fourier-2_3-real_phi',
      'split':[N_TRAIN,N_VAL,N_TEST]}
SPEC_SHA=hashlib.sha256(json.dumps(SPEC,sort_keys=True).encode()).hexdigest()[:16]
print('Dataset:',DATA_SHAPE,'~',round(np.prod(DATA_SHAPE)*4/2**30,3),'GiB')
print('Split: train [0,200), val [200,230), test [230,300)')
print('Specification:',SPEC_SHA,'Drive output:',ROOT)

In [ ]:
#@title 2. Self-contained spectral ETDRK4 solver (GPU) and deterministic initial data

def seed_vorticity(index,n=NX):
    """Same IC family/seed convention as the uploaded NumPy CPU generator."""
    rng=np.random.default_rng(BASE_SEED+9973*int(index))
    eta=rng.standard_normal((n,n))
    freq=2*np.pi*np.fft.fftfreq(n,d=DOMAIN/n)
    kx,ky=np.meshgrid(freq,freq,indexing='ij')
    K=np.sqrt(kx*kx+ky*ky)
    filt=np.exp(-(K/IC_FILTER_K0)**4)
    filt[0,0]=0.0
    dealias=(np.abs(kx)<=n//3)*(np.abs(ky)<=n//3)
    w=np.fft.ifft2(np.fft.fft2(eta)*filt*dealias).real
    w-=w.mean()
    st=w.std()
    if st>0:w=(IC_AMPLITUDE/st)*w
    return w.astype(np.float32)


def phi_series(z,j,terms=32):
    """phi_j(z) = sum(z**k/(k+j)!, k>=0); stable at z=0.

    z real float64, j=1,2,3. This avoids catastrophic cancellation.
    For this benchmark max |dt*nu*k²| <=~0.33 even at 256².
    """
    term=torch.ones_like(z)/math.factorial(j)
    result=term.clone()
    for k in range(1,terms+1):
        term=term*z/(k+j)
        result=result+term
    return result

class KolmogorovETDRK4:
    def __init__(self,n=128,dt=0.005,dtype=torch.float32,device=DEVICE):
        assert n%2==0 and n>=16 and n%4==0
        self.n=n; self.dt=dt; self.dtype=dtype; self.device=device
        self.cd=torch.complex64 if dtype==torch.float32 else torch.complex128
        k=2*np.pi*torch.fft.fftfreq(n,d=float(DOMAIN)/n,device=device,dtype=torch.float64)
        kx,ky=torch.meshgrid(k,k,indexing='ij')
        k2=kx*kx+ky*ky
        cut=n//3
        self.cut=cut
        mask=(torch.abs(kx)<=cut)&(torch.abs(ky)<=cut)
        self.mask=mask.to(self.cd)
        self.kx=kx.to(dtype);self.ky=ky.to(dtype)
        self.safe_k2=k2.clamp_min(1.0).to(dtype)
        self.safe_k2[0,0]=1.
        self.kmag=torch.sqrt(k2).to(dtype)
        y=(torch.arange(n,device=device,dtype=dtype)*float(DOMAIN)/n)[None,:]
        source=-FORCING_MODE*torch.cos(FORCING_MODE*y).expand(n,-1)
        self.source_hat=(torch.fft.fft2(source)*self.mask).to(self.cd)
        self.source_hat[0,0]=0
        z=-NU*k2*dt
        self.E=torch.exp(z).to(dtype)
        self.E2=torch.exp(z/2).to(dtype)
        p1,p2,p3=(phi_series(z,i) for i in (1,2,3))
        self.Q=(.5*dt*phi_series(z/2,1)).to(dtype)
        self.F1=(dt*(p1-3*p2+4*p3)).to(dtype)
        self.F2=(dt*(p2-2*p3)).to(dtype)
        self.F3=(dt*(4*p3-p2)).to(dtype)
        # At zero frequency RK4 coefficients must each equal dt/6.
        assert abs(float(self.F1[0,0])-dt/6)<1e-7
        assert abs(float(self.F2[0,0])-dt/6)<1e-7
        assert abs(float(self.F3[0,0])-dt/6)<1e-7

    def to_spectral(self,w):
        z=torch.fft.fft2(w.to(device=self.device,dtype=self.dtype)) * self.mask
        z[...,0,0]=0
        return z

    def physical(self,z):
        return torch.fft.ifft2(z).real

    def velocities(self,z):
        psi=z/self.safe_k2
        u=torch.fft.ifft2((1j*self.ky)*psi).real
        v=torch.fft.ifft2((-1j*self.kx)*psi).real
        return u,v

    def nonlinearity(self,z):
        z=z*self.mask
        u,v=self.velocities(z)
        dx=torch.fft.ifft2((1j*self.kx)*z).real
        dy=torch.fft.ifft2((1j*self.ky)*z).real
        nonlinear=(-torch.fft.fft2(u*dx+v*dy)+self.source_hat)*self.mask
        nonlinear[...,0,0]=0
        return nonlinear

    def step(self,z):
        Nv=self.nonlinearity(z)
        a=self.E2*z+self.Q*Nv
        Na=self.nonlinearity(a)
        b=self.E2*z+self.Q*Na
        Nb=self.nonlinearity(b)
        c=self.E2*a+self.Q*(2*Nb-Nv)
        Nc=self.nonlinearity(c)
        result=(self.E*z+self.F1*Nv+2*self.F2*(Na+Nb)+self.F3*Nc)*self.mask
        result[...,0,0]=0
        return result

    def cfl(self,z):
        u,v=self.velocities(z)
        factor=self.dt/(DOMAIN/self.n)
        return float((u.abs().amax(dim=(-2,-1)) + v.abs().amax(dim=(-2,-1))).max().item()*factor)

    def shell_energy_fraction(self,z,lower_fraction=0.8):
        power=z.abs().square().sum(dim=0)
        region=(torch.maximum(self.kx.abs(),self.ky.abs())>=lower_fraction*self.cut)
        return float((power[region].sum()/power.sum().clamp_min(1e-30)).item())

    @torch.no_grad()
    def advance(self,z,steps,monitor=False,check_every=250):
        maxcfl=0.0
        for k in range(1,steps+1):
            z=self.step(z)
            if monitor and (k==1 or k%check_every==0 or k==steps):
                assert bool(torch.isfinite(z).all()), f'Nonfinite spectral state at step {k}'
                cfl=self.cfl(z)
                maxcfl=max(maxcfl,cfl)
                if cfl>MAX_RECORDED_CFL:
                    raise RuntimeError(f'CFL={cfl:.3f} exceeds gate {MAX_RECORDED_CFL}. '
                                       'Reduce DT and rerun convergence tests.')
        return z,maxcfl


def spectral_resample(w,target_n):
    """FFT-orthogonal interpolation / restriction preserving Fourier amplitudes.

    Input assumed real-valued and bandlimited below old Nyquist. We use
    fft2(..., norm='forward'), centered Fourier-domain pad/crop, and
    ifft2(..., norm='forward') so both grid sizes represent same field.
    """
    old=w.shape[-1]
    assert old==w.shape[-2] and old%2==0 and target_n%2==0
    z=torch.fft.fftshift(torch.fft.fft2(w,norm='forward'),dim=(-2,-1))
    if target_n>old:
        p=(target_n-old)//2
        z=torch.nn.functional.pad(z,(p,p,p,p))
    elif target_n<old:
        p=(old-target_n)//2
        z=z[...,p:p+target_n,p:p+target_n]
    return torch.fft.ifft2(torch.fft.ifftshift(z,dim=(-2,-1)),norm='forward').real


def rel_field(a,b):
    """Combined relative field L2 over all samples, nodes, time."""
    return float((torch.linalg.vector_norm((a-b).reshape(-1).double())/
                 torch.linalg.vector_norm(b.reshape(-1).double()).clamp_min(1e-20)).item())

print('GPU solver ready. No data generated by this cell.')

In [ ]:
#@title 3. Mandatory numerical verification (no full dataset generation yet)

@torch.no_grad()
def validate_solver():
    stamp=time.perf_counter()
    report={'specification_sha256_prefix':SPEC_SHA,
            'gpu':torch.cuda.get_device_name(0),'dtype':'float32/complex64',
            'gates':{'dt_rel':DT_REFINEMENT_MAX_REL,
                     'space_rel':SPATIAL_REFINEMENT_MAX_REL,
                     'fp_rel':FP32_FP64_MAX_REL,
                     'cfl':MAX_RECORDED_CFL,
                     'shell':MAX_SPECTRAL_CUTOFF_SHELL_FRACTION}}
    gpu=KolmogorovETDRK4(NX,DT,torch.float32,DEVICE)

    # Analytical forced laminar solution: omega=A(t)cos(4y), u·grad omega = 0.
    y=torch.arange(NX,device=DEVICE,dtype=torch.float32)*(DOMAIN/NX)
    w=(2*torch.cos(FORCING_MODE*y)).expand(NX,-1)[None].clone()
    z=gpu.to_spectral(w)
    steps=100
    z,_=gpu.advance(z,steps)
    alpha0=2.0
    lam=NU*FORCING_MODE**2
    alpha=alpha0*np.exp(-lam*(DT*steps))-(FORCING_MODE/lam)*(1-np.exp(-lam*(DT*steps)))
    exact=(alpha*torch.cos(FORCING_MODE*y)).expand(NX,-1)[None]
    analytic_error=rel_field(gpu.physical(z),exact)
    report['laminar_analytic_relative_error']=analytic_error
    print(f'[ANALYTIC] forced laminar relative error {analytic_error:.3e}')
    assert analytic_error<0.001, 'Analytic solver verification FAILED.'

    # Verify FFT restriction/interpolation is identity on a properly bandlimited field.
    initial=torch.from_numpy(np.stack([seed_vorticity(i) for i in range(CHECK_SAMPLES)])).to(DEVICE)
    initial=gpu.physical(gpu.to_spectral(initial))
    resample_error=rel_field(spectral_resample(spectral_resample(initial,256),NX),initial)
    report['spectral_resampling_roundtrip_relative_error']=resample_error
    assert resample_error<1e-5

    # Spinup used only to form physically representative initial states.
    t0=time.perf_counter()
    z0=gpu.to_spectral(initial)
    z_spin,cfl_spin=gpu.advance(z0,SPINUP_STEPS,monitor=True,check_every=250)
    spin_states=gpu.physical(z_spin).detach()
    shell=gpu.shell_energy_fraction(z_spin)
    report['spinup_seconds_for_'+str(CHECK_SAMPLES)+'_trajectories']=time.perf_counter()-t0
    report['max_spinup_cfl']=cfl_spin
    report['spinup_spectral_cutoff_shell_fraction']=shell
    print(f'[SPINUP] {CHECK_SAMPLES} trajectories, {report["spinup_seconds_for_"+str(CHECK_SAMPLES)+"_trajectories"]:.1f}s, '
          f'CFLmax={cfl_spin:.3f}, outer resolved spectral shell energy={shell:.4%}')
    assert shell<MAX_SPECTRAL_CUTOFF_SHELL_FRACTION, (
        'Spectral energy near the 128² dealiased cutoff is too large. '
        'Try higher spatial resolution before publishing data.')

    # Independent nonlinear physics check: incompressible advection cannot
    # change the spatially integrated enstrophy of a dealiased periodic field.
    # It also cannot alter the spatial mean of vorticity.
    random_z=gpu.to_spectral(initial)
    adv_hat=gpu.nonlinearity(random_z)-gpu.source_hat
    adv_phys=gpu.physical(adv_hat)
    random_phys=gpu.physical(random_z)
    advection_mean=float(adv_phys.mean(dim=(-2,-1)).abs().max().item())
    adv_enstrophy=(random_phys*adv_phys).mean(dim=(-2,-1)).abs()
    denom=torch.sqrt((random_phys.square().mean(dim=(-2,-1)) *
                      adv_phys.square().mean(dim=(-2,-1))).clamp_min(1e-20))
    relative_advection_enstrophy=float((adv_enstrophy/denom).max().item())
    report['relative_enstrophy_advection_defect']=relative_advection_enstrophy
    report['mean_advection_defect']=advection_mean
    print(f'[PHYSICS] advection mean defect={advection_mean:.2e}, '
          f'relative enstrophy defect={relative_advection_enstrophy:.2e}')
    assert advection_mean<1e-5 and relative_advection_enstrophy<1e-4, (
        'Nonlinear advection does not preserve the expected periodic mean/'
        'enstrophy identity. Check velocity recovery, dealiasing, and FFT axes.')

    # Temporal refinement: same post-spinup initial conditions; 200 vs 400 steps.
    half=KolmogorovETDRK4(NX,DT/2,torch.float32,DEVICE)
    coarse0=gpu.to_spectral(spin_states)
    fine0=half.to_spectral(spin_states)
    coarse,cfl_coarse=gpu.advance(coarse0,FORECAST_STEPS,monitor=True,check_every=50)
    fine,cfl_fine=half.advance(fine0,2*FORECAST_STEPS,monitor=True,check_every=100)
    t_rel=rel_field(gpu.physical(coarse),half.physical(fine))
    report['dt_halving_final_relative_error']=t_rel
    report['max_forecast_cfl']=max(cfl_coarse,cfl_fine)
    print(f'[TIME] dt={DT} vs {DT/2}: final relative L2={t_rel:.4%}')
    assert t_rel<DT_REFINEMENT_MAX_REL, (
        'Temporal refinement FAILED; decrease DT and regenerate at a finer solver step.')

    # 128² vs 256², from IDENTICAL initial physical fields, same dt/2.
    high=KolmogorovETDRK4(256,DT/2,torch.float32,DEVICE)
    z256=high.to_spectral(spectral_resample(spin_states,256))
    fine256,cfl256=high.advance(z256,2*FORECAST_STEPS,monitor=True,check_every=100)
    proj128=spectral_resample(high.physical(fine256),NX)
    s_rel=rel_field(half.physical(fine),proj128)
    report['128_vs_256_final_relative_error']=s_rel
    report['max_refined_grid_cfl']=cfl256
    print(f'[SPACE] 128² vs 256² at dt/2: projected final relative L2={s_rel:.4%}')
    assert s_rel<SPATIAL_REFINEMENT_MAX_REL, (
        'Spatial refinement FAILED: 128² is not adequately demonstrated; '
        'inspect higher-resolution data before training AWFNO.')

    # Short-horizon, independently computed float64 GPU reference (FP64 on
    # workstation GPUs can be slow; we check 20 steps, not all spinup steps).
    d64=KolmogorovETDRK4(NX,DT,torch.float64,DEVICE)
    z32,_=gpu.advance(gpu.to_spectral(spin_states[:1]),20)
    z64,_=d64.advance(d64.to_spectral(spin_states[:1].double()),20)
    fp_rel=rel_field(gpu.physical(z32),d64.physical(z64))
    report['fp32_vs_fp64_short_horizon_relative_error']=fp_rel
    print(f'[PRECISION] 20-step fp32 vs fp64 relative error={fp_rel:.3e}')
    assert fp_rel<FP32_FP64_MAX_REL, 'Float32 solver precision check FAILED.'

    # Low-cost GPU throughput pilot, before committing to 300 trajectories.
    # Tuning only determines expected billing time, not an error criterion.
    benchmark_count=min(BATCH_SIZE,N_SAMPLES)
    bench_ics=torch.from_numpy(np.stack([seed_vorticity(i)
                                         for i in range(benchmark_count)])).to(DEVICE)
    zz=gpu.to_spectral(bench_ics)
    for _ in range(5):zz=gpu.step(zz)
    torch.cuda.synchronize()
    pilot_steps=40
    pilot_start=time.perf_counter()
    for _ in range(pilot_steps):zz=gpu.step(zz)
    torch.cuda.synchronize()
    seconds_per_step=(time.perf_counter()-pilot_start)/pilot_steps
    estimated_seconds=seconds_per_step*(SPINUP_STEPS+FORECAST_STEPS)*math.ceil(N_SAMPLES/BATCH_SIZE)
    report['pilot_gpu_seconds_per_batched_step']=seconds_per_step
    report['rough_total_gpu_generation_hours']=estimated_seconds/3600
    print(f'[COST PILOT] {benchmark_count} trajectories/batch, '
          f'{seconds_per_step:.4f}s/step; rough GPU evolution estimate '
          f'{estimated_seconds/3600:.2f}h (+ Drive writes/checks).')
    if estimated_seconds/3600 > MAX_GENERATION_HOURS:
        raise RuntimeError('Estimated generation runtime exceeds cost guard '
                           f'{MAX_GENERATION_HOURS}h. Inspect GPU, BATCH_SIZE and '
                           'pilot results before spending money.')
    del bench_ics,zz

    report['total_check_seconds']=time.perf_counter()-stamp
    report['status']='PASS'
    check_path=ROOT/f'numerical_verification_{SPEC_SHA}.json'
    tmp=LOCAL/'numerical_verification.json'
    tmp.write_text(json.dumps(report,indent=2))
    shutil.copy2(tmp,check_path)
    print('\nALL NUMERICAL GATES PASSED in',round(report['total_check_seconds'],1),'seconds')
    print('Numerical verification saved:',check_path)
    return report

VERIFICATION=validate_solver()

In [ ]:
#@title 4. Generate the 300 independent trajectories in resumable GPU batches
# Individual complete batches are stored on Drive. Identical specification
# and deterministic trajectory IDs mean re-running this cell safely resumes.

assert VERIFICATION['status']=='PASS', 'Run the numerical verification cell first.'
config_path=ROOT/'dataset_specification.json'
if config_path.exists():
    old=json.loads(config_path.read_text())
    if old['spec_sha']!=SPEC_SHA or old['spec']!=SPEC:
        raise RuntimeError('OUTPUT FOLDER CONFLICT: contains a different dataset specification. '
                           'Choose a new DRIVE_FOLDER. Existing data are untouched.')
else:
    src=LOCAL/'dataset_specification.json'
    src.write_text(json.dumps({'spec_sha':SPEC_SHA,'spec':SPEC},indent=2))
    shutil.copy2(src,config_path)

@torch.no_grad()
def create_chunk(begin,end,engine):
    samples=np.stack([seed_vorticity(i) for i in range(begin,end)],axis=0)
    z=engine.to_spectral(torch.from_numpy(samples).to(DEVICE))
    z,max_spinup_cfl=engine.advance(z,SPINUP_STEPS,monitor=True,check_every=250)
    omega_out=np.empty((end-begin,NX,NX,NT),dtype=np.float32)
    max_forecast_cfl=0.0
    for st in range(FORECAST_STEPS+1):
        if st%SAVE_EVERY==0:
            frame=engine.physical(z)
            assert bool(torch.isfinite(frame).all()),f'Nonfinite snapshot at {begin}:{end}, step {st}'
            omega_out[...,st//SAVE_EVERY]=frame.cpu().numpy()
        if st<FORECAST_STEPS:
            z=engine.step(z)
        if st>0 and st%50==0:
            assert bool(torch.isfinite(z).all()),f'Nonfinite state at step {st}'
            cc=engine.cfl(z)
            max_forecast_cfl=max(max_forecast_cfl,cc)
            if cc>MAX_RECORDED_CFL:
                raise RuntimeError(f'CFL {cc:.3f} > {MAX_RECORDED_CFL}; generation aborted.')
    means=np.max(np.abs(omega_out.mean(axis=(1,2))))
    if means>0.002:
        raise RuntimeError(f'Nonzero mean vorticity ({means:.3g})')
    change=np.sqrt(np.mean((omega_out[...,-1]-omega_out[...,0])**2,axis=(1,2)))
    if np.any(change<1e-4):
        raise RuntimeError('A trajectory is nearly time-independent; investigate the generator.')
    return omega_out,{'max_spinup_cfl':max_spinup_cfl,
                      'max_forecast_cfl':max_forecast_cfl,
                      'max_absolute_spatial_mean':float(means),
                      'min_rms_first_last_change':float(np.min(change)),
                      'min':float(omega_out.min()),'max':float(omega_out.max()),
                      'std':float(omega_out.std())}


def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(8*1024*1024)
            if not b:break
            h.update(b)
    return h.hexdigest()


def chunk_paths(begin,end):
    stem=f'{SPEC_SHA}_samples_{begin:03d}_{end-1:03d}'
    return CHUNKS/(stem+'.npy'), CHUNKS/(stem+'.json')

engine=KolmogorovETDRK4(NX,DT,torch.float32,DEVICE)
num_chunks=math.ceil(N_SAMPLES/BATCH_SIZE)
start_total=time.perf_counter()
for ci,begin in enumerate(range(0,N_SAMPLES,BATCH_SIZE)):
    end=min(begin+BATCH_SIZE,N_SAMPLES)
    target,info_path=chunk_paths(begin,end)
    t_start=time.perf_counter()
    if target.exists() and info_path.exists():
        info=json.loads(info_path.read_text())
        expected=[end-begin,NX,NX,NT]
        assert info['spec_sha']==SPEC_SHA and info['shape']==expected
        checksum=sha256_file(target)
        assert checksum==info['sha256'],f'Corrupted chunk {target}; remove manually and regenerate.'
        print(f'[RESUME] batch {ci+1}/{num_chunks} IDs {begin}:{end}, checksum OK',flush=True)
        continue
    if target.exists() or info_path.exists():
        raise RuntimeError(f'Partial/inconsistent batch files: {target}, {info_path}. '
                           'Inspect them before manual removal; existing data are not overwritten.')
    t0=time.perf_counter()
    chunk,diag=create_chunk(begin,end,engine)
    local_arr=LOCAL/target.name
    np.save(local_arr,chunk,allow_pickle=False)
    checksum=sha256_file(local_arr)
    local_meta=LOCAL/info_path.name
    local_meta.write_text(json.dumps({
        'spec_sha':SPEC_SHA,'begin':begin,'end':end,
        'shape':list(chunk.shape),'dtype':str(chunk.dtype),'sha256':checksum,
        'generator_seconds':time.perf_counter()-t0,'diagnostics':diag
    },indent=2))
    shutil.copy2(local_arr,target)
    if sha256_file(target)!=checksum:raise IOError('Drive copy checksum mismatch: '+str(target))
    # JSON metadata is written last to mark this chunk committed.
    shutil.copy2(local_meta,info_path)
    del chunk
    local_arr.unlink();local_meta.unlink()
    gc.collect()
    elapsed=time.perf_counter()-start_total
    predicted_remaining=elapsed/(ci+1)*max(num_chunks-ci-1,0)
    print(f'[SAVED] batch {ci+1}/{num_chunks} IDs {begin}:{end}, '
          f'batch={time.perf_counter()-t_start:.1f}s, '
          f'elapsed={elapsed/60:.1f}min, rough remaining={predicted_remaining/60:.1f}min',flush=True)
print('All trajectory batch files are present and individually checksummed.')

In [ ]:
#@title 5. Assemble verified PyTorch dataset and save it to Google Drive
assert VERIFICATION['status']=='PASS'
np_path=LOCAL/'kolmogorov_omega.npy'
out=np.lib.format.open_memmap(np_path,mode='w+',dtype=np.float32,shape=DATA_SHAPE)
chunk_log=[]
for begin in range(0,N_SAMPLES,BATCH_SIZE):
    end=min(begin+BATCH_SIZE,N_SAMPLES)
    src,info_path=chunk_paths(begin,end)
    assert src.exists() and info_path.exists(),f'Missing chunk {src}'
    info=json.loads(info_path.read_text())
    assert info['spec_sha']==SPEC_SHA
    assert sha256_file(src)==info['sha256'],'Drive chunk checksum mismatch: '+str(src)
    part=np.load(src,mmap_mode='r',allow_pickle=False)
    assert part.shape==(end-begin,NX,NX,NT) and part.dtype==np.float32
    assert np.isfinite(part).all(),'Non-finite data in chunk'
    out[begin:end]=part
    chunk_log.append(info)
out.flush()

# Exact independent ICs, evolving fields, valid nonoverlapping split and time grid.
train_idx=np.arange(0,N_TRAIN,dtype=np.int64)
val_idx=np.arange(N_TRAIN,N_TRAIN+N_VAL,dtype=np.int64)
test_idx=np.arange(N_TRAIN+N_VAL,N_SAMPLES,dtype=np.int64)
assert len(set(train_idx)|set(val_idx)|set(test_idx))==N_SAMPLES
assert not(set(train_idx)&set(val_idx) or set(train_idx)&set(test_idx) or set(val_idx)&set(test_idx))
first_frames=np.asarray(out[...,0])
initial_hashes=[hashlib.sha256(np.ascontiguousarray(first_frames[i]).tobytes()).hexdigest()
                for i in range(N_SAMPLES)]
assert len(set(initial_hashes))==N_SAMPLES,'Duplicate post-spinup initial states!'
assert np.isfinite(out).all()
max_mean=max(float(np.abs(np.mean(out[i],axis=(0,1))).max()) for i in range(N_SAMPLES))
assert max_mean<=0.002,f'Mean vorticity too large: {max_mean}'

# The original benchmark uses exactly these field and metadata keys.
# Both U and omega intentionally point to the SAME tensor storage.
tensor=torch.from_numpy(out)
payload={
  'U':tensor,'omega':tensor,
  'x':torch.from_numpy(X.astype(np.float32)),
  'y':torch.from_numpy(Y.astype(np.float32)),
  'times':torch.from_numpy(TIMES.astype(np.float32)),
  'train_indices':torch.from_numpy(train_idx),
  'val_indices':torch.from_numpy(val_idx),
  'test_indices':torch.from_numpy(test_idx),
  'Re':RE,'nu':NU,'forcing_mode':FORCING_MODE,
  'dt':DT,'T_spinup':T_SPINUP,'T_forecast':T_FORECAST,
  'axis_order':'sample, x, y, time',
  'generator_spec_sha':SPEC_SHA,
  'method':'real-coefficient ETDRK4 Fourier pseudospectral 2/3-dealiased',
}
local_pt=LOCAL/'kolmogorov_dataset.pt'
start=time.perf_counter()
torch.save(payload,local_pt)
print('Local .pt:',local_pt,'GiB:',round(local_pt.stat().st_size/2**30,3),
      'serialization seconds:',round(time.perf_counter()-start,1))
# Verify we can actually reload from disk before exposing the dataset.
try:
    verify=torch.load(local_pt,map_location='cpu',weights_only=True,mmap=True)
except TypeError:
    verify=torch.load(local_pt,map_location='cpu',weights_only=True)
assert tuple(verify['U'].shape)==DATA_SHAPE
assert verify['U'].data_ptr()==verify['omega'].data_ptr()
assert len(verify['val_indices'])==N_VAL
assert bool(torch.isfinite(verify['U']).all())
del verify,payload,tensor,out,first_frames
gc.collect()

pt_sha=sha256_file(local_pt)
final=ROOT/'kolmogorov_dataset.pt'
if final.exists():
    if sha256_file(final)!=pt_sha:
        raise RuntimeError('Existing final file differs. Choose a new DRIVE_FOLDER; '
                           'do not silently overwrite a previously published dataset.')
    print('[RESUME] Final Drive dataset already exists with identical SHA256.')
else:
    temporary=ROOT/'kolmogorov_dataset.pt.partial'
    if temporary.exists():
        temporary.unlink()  # incomplete prior upload; completed chunk data are safe
    shutil.copy2(local_pt,temporary)
    assert sha256_file(temporary)==pt_sha,'Drive copy verification FAILED'
    os.replace(temporary,final)
    assert sha256_file(final)==pt_sha,'Final Drive checksum FAILED'

summary={
 'status':'PASS','spec_sha':SPEC_SHA,'spec':SPEC,'final_name':final.name,
 'final_bytes':final.stat().st_size,'final_sha256':pt_sha,
 'full_dataset_shape':list(DATA_SHAPE),
 'split_train_indices':train_idx.tolist(),
 'split_val_indices':val_idx.tolist(),
 'split_test_indices':test_idx.tolist(),
 'max_abs_spatial_mean_vorticity':max_mean,
 'independent_initial_state_count':len(set(initial_hashes)),
 'numerical_verification':VERIFICATION,
 'chunks':chunk_log,
}
meta_local=LOCAL/'metadata.json'
meta_local.write_text(json.dumps(summary,indent=2))
shutil.copy2(meta_local,ROOT/'metadata.json')
print('\nVERIFIED DATASET STORED IN GOOGLE DRIVE:',final)
print('Dataset SHA256:',pt_sha)
print('Metadata:',ROOT/'metadata.json')
if REMOVE_CHUNKS_AFTER_FINAL_VERIFICATION:
    # Only remove intermediate batches after the final .pt and metadata pass checks.
    for begin in range(0,N_SAMPLES,BATCH_SIZE):
        end=min(begin+BATCH_SIZE,N_SAMPLES)
        cf,jf=chunk_paths(begin,end)
        cf.unlink();jf.unlink()
    print('Intermediate Drive chunks removed after final verification.')

In [ ]:
#@title 6. Final report and snapshot preview (does not train any ML model)
assert (ROOT/'kolmogorov_dataset.pt').exists()
print('Dataset:',ROOT/'kolmogorov_dataset.pt')
print('Dataset size GiB:',round((ROOT/'kolmogorov_dataset.pt').stat().st_size/2**30,3))
print('Shape:',DATA_SHAPE,'dtype=float32')
print('Splits: 200 training / 30 validation / 70 test')
print('Forecast times:',TIMES[[0,1,5,10,25,50]])
print('Numerical time refinement final relative error:',
      f'{VERIFICATION["dt_halving_final_relative_error"]:.4%}')
print('128-vs-256 spatial refinement final relative error:',
      f'{VERIFICATION["128_vs_256_final_relative_error"]:.4%}')
print('Max spinup CFL:',round(VERIFICATION['max_spinup_cfl'],3))
print('Saved SHA256:',json.loads((ROOT/'metadata.json').read_text())['final_sha256'])

source=np.load(LOCAL/'kolmogorov_omega.npy',mmap_mode='r')
fig,axes=plt.subplots(1,4,figsize=(15,3.6),constrained_layout=True)
ids=[0,10,25,50]
vmin,vmax=float(source[0].min()),float(source[0].max())
for ax,index in zip(axes,ids):
    ax.imshow(source[0,:,:,index].T,origin='lower',
              extent=[0,2*np.pi,0,2*np.pi],vmin=vmin,vmax=vmax,cmap='RdBu_r')
    ax.set_title(f't={TIMES[index]:.2f}');ax.set_xlabel('x')
    ax.set_ylabel('y')
path=ROOT/'preview_trajectory_000.png'
fig.savefig(LOCAL/'preview_trajectory_000.png',dpi=160)
shutil.copy2(LOCAL/'preview_trajectory_000.png',path)
plt.show()
print('\nSUCCESS. You can now use kolmogorov_dataset.pt in the separate AWFNO notebook.')